# 02 – Incremental load from PostgreSQL into the lake (`append`)

**Goal (Task 2, part 3):** load only **new or changed rows** since the last run, and **add** them to the lake (mode `append`).

**How dlt knows what is new:** we pick an **index (cursor) column**, `updated_at`. dlt remembers the **highest `updated_at`** it has loaded (its *state*). Next run it asks the database only for rows with `updated_at` greater than that value.

**Why `updated_at` and not `id`?**
- `id` only grows → it catches **new** rows but never **edited** rows.
- `updated_at` changes on **every insert and every update** (a database trigger sets it, see `backend/migrations/001_add_updated_at.sql`) → it catches **new and edited** rows.

The incremental copies go to separate folders (`postgres/animals_incremental/...`) so they don't mix with the full copies from notebook 01 (a `replace` run would delete them).

In [ ]:
import dlt
import pandas as pd
import sqlalchemy as sa
from dlt.sources.sql_database import sql_table
from lake import (POSTGRES_URL, DB_TABLES, PIPELINES_DIR,
                  lake_destination, add_metadata, list_lake, read_lake, ensure_bucket)

engine = sa.create_engine(POSTGRES_URL)   # used only to look at / change the database in the demo cells

## Step 1 – One incremental resource per table

- `sql_table(..., incremental=dlt.sources.incremental("updated_at"))` → read only rows newer than the last run
- `apply_hints(table_name=...)` → name in the lake: `<table>_incremental`
- `add_metadata(...)` → same 5 metadata columns, `_ingestion_type = "incremental"`

In [ ]:
def build_incremental_resources():
    resources = []
    for name in DB_TABLES:
        r = sql_table(POSTGRES_URL, table=name,
                      incremental=dlt.sources.incremental("updated_at"))
        r.apply_hints(table_name=f"{name}_incremental")
        add_metadata(r, source_system="postgres", source_resource=name, ingestion_type="incremental")
        resources.append(r)
    return resources


def run_incremental(pipeline):
    pipeline.run(build_incremental_resources(), write_disposition="append", loader_file_format="parquet")
    counts = pipeline.last_trace.last_normalize_info.row_counts
    print("New rows loaded:", {k: v for k, v in counts.items() if not k.startswith("_dlt")} or "none")


incr_pipeline = dlt.pipeline(
    pipeline_name="pawhome_incremental",
    destination=lake_destination("silo"),
    dataset_name="postgres",
    pipelines_dir=PIPELINES_DIR,
)

## Step 2 – First run: everything is "new"

The first time, dlt has no state yet, so it loads **all** rows.

In [ ]:
run_incremental(incr_pipeline)
list_lake("postgres/animals_incremental")

## Step 3 – Run again without changing anything

Nothing changed in the database → **0 new rows**, **no new files**.

In [ ]:
run_incremental(incr_pipeline)
list_lake("postgres/animals_incremental")

## Step 4 – Add some records, then run again

**Option A (recommended for the demo):** open the PawHome app (http://localhost:5173) and add 2–3 animals. Then **skip** the "Option B" cell below.

**Option B:** run the next cell, which inserts 3 animals with SQL.

Then run the incremental load: only the **new** rows arrive, in a **new** small file.

In [ ]:
# Option B (skip this cell if you added animals in the app)
with engine.begin() as conn:
    new_ids = conn.execute(sa.text("""
        INSERT INTO animals (shelter_id, name, species, breed, sex, age_years, size, arrived_at)
        VALUES (1, 'Lake Test 1', 'dog', 'Beagle', 'M', 2, 'medium', CURRENT_DATE),
               (1, 'Lake Test 2', 'cat', 'Persian', 'F', 1, 'small', CURRENT_DATE),
               (2, 'Lake Test 3', 'rabbit', 'Holland Lop', 'F', 0.5, 'small', CURRENT_DATE)
        RETURNING id""")).scalars().all()
print("Inserted animal ids:", new_ids)

In [ ]:
run_incremental(incr_pipeline)
list_lake("postgres/animals_incremental")

In [ ]:
# The rows that arrived in the LAST run
lake_animals = read_lake("postgres/animals_incremental/")
last_load = lake_animals["_load_id"].astype(float).max()
lake_animals[lake_animals["_load_id"].astype(float) == last_load][["id", "name", "updated_at", "_load_id"]]

## Step 5 – Update and delete records, then run again

**Option A (recommended):** in the app, edit one animal (e.g. change its name) and delete another one. Then **skip** the "Option B" cell below.

**Option B:** the next cell renames the newest animal and deletes the second-newest one.

In [ ]:
# Option B (skip this cell if you edited and deleted in the app)
with engine.begin() as conn:
    updated_id, deleted_id = conn.execute(sa.text("SELECT id FROM animals ORDER BY id DESC LIMIT 2")).scalars().all()
    conn.execute(sa.text("UPDATE animals SET name = name || ' (renamed)' WHERE id = :id"), {"id": updated_id})
    conn.execute(sa.text("DELETE FROM animals WHERE id = :id"), {"id": deleted_id})
print("Updated animal", updated_id, "and deleted animal", deleted_id)

In [ ]:
run_incremental(incr_pipeline)
lake_animals = read_lake("postgres/animals_incremental/")

### What happened to the UPDATE?

Every edited animal appears in the lake **more than once**: the old version and the new version (newer `updated_at`, different `_load_id`). `append` never changes old files, it only **adds**. The lake keeps the **history** of the row.

In [ ]:
# Animals that appear more than once in the lake = animals that were updated
versions = lake_animals[lake_animals.duplicated("id", keep=False)].sort_values(["id", "updated_at"])
versions[["id", "name", "updated_at", "_load_id", "_ingested_at"]]

To get the **current** state from the lake, keep only the newest version of each `id`:

In [ ]:
latest = (lake_animals.sort_values("updated_at")
                      .drop_duplicates(subset="id", keep="last"))
print("rows in lake:", len(lake_animals), "→ latest versions:", len(latest))
latest[latest.id.isin(versions.id)][["id", "name", "updated_at"]]

### What happened to the DELETE?

**Nothing.** A deleted row no longer exists in the database, so there is no row with a newer `updated_at` for the incremental query to find. The lake still contains the deleted animal.

In [ ]:
db_ids = set(pd.read_sql("SELECT id FROM animals", engine).id)
in_lake_not_in_db = latest[~latest.id.isin(db_ids)]
print("Deleted in the database but still in the lake:")
in_lake_not_in_db[["id", "name", "updated_at"]]

**Conclusion**

| Change in the app | Visible in the lake with incremental `append` on `updated_at`? |
|---|---|
| **INSERT** | ✅ yes – new rows are added |
| **UPDATE** | ✅ yes – as a **new version** of the row (old version stays → history, duplicates by `id`) |
| **DELETE** | ❌ no – the deleted row stays in the lake |

How deletes could be captured: **soft delete** (a `deleted_at` column instead of really deleting, which also changes `updated_at`), **CDC** (change data capture from the database log), or a periodic **full load** to compare against.

Also: if the cursor were `id` instead of `updated_at`, updates would **not** be visible either.

## Bonus – Incremental load into the additional lake (RustFS)

The same resources, a **second pipeline** with its own state, writing into RustFS. The first run loads everything, later runs only changes.

In [ ]:
ensure_bucket("rustfs")

incr_pipeline_lake2 = dlt.pipeline(
    pipeline_name="pawhome_incremental_rustfs",
    destination=lake_destination("rustfs"),
    dataset_name="postgres",
    pipelines_dir=PIPELINES_DIR,
)
run_incremental(incr_pipeline_lake2)
list_lake("postgres", lake="rustfs")

In [ ]:
# Run again: nothing new → no new files
run_incremental(incr_pipeline_lake2)